# គំរូ 02: OpenAI SDK Integration

កំណត់ត្រានេះបង្ហាញពីការរួមបញ្ចូលកម្រិតខ្ពស់ជាមួយ OpenAI Python SDK ដែលគាំទ្រទាំង Microsoft Foundry Local និង Azure OpenAI ជាមួយចម្លើយជាស្ទ្រីម និងការដោះស្រាយកំហុសយ៉ាងត្រឹមត្រូវ។

## ទិដ្ឋភាពទូទៅ

គំរូនេះបង្ហាញ៖
- ការប្តូរយ៉ាងរហ័សរលូនរវាង Foundry Local និង Azure OpenAI
- ការផ្តល់ចម្លើយជាស្ទ្រីមសម្រាប់បទពិសោធន៍អ្នកប្រើកាន់តែល្អ
- ការប្រើប្រាស់ត្រឹមត្រូវនៃ FoundryLocalManager SDK
- ការដោះស្រាយកំហុសយ៉ាងរឹងមាំ និងមេកានិចជំនួស
- លំនាំកូដដែលរួចរាល់សម្រាប់ផលិតកម្ម


## លក្ខខណ្ឌជាមុន

- **Foundry Local**: បានតំឡើង និងកំពុងដំណើរការ (សម្រាប់ការសន្និដ្ឋានក្នុងស្រុក)
- **Python**: 3.8 ឬថ្មីជាងនេះ ជាមួយ OpenAI SDK
- **Azure OpenAI**: Endpoint និង API key ត្រឹមត្រូវ (សម្រាប់ការសន្និដ្ឋានក្នុងពពក)

### តំឡើងផ្នែកដែលត្រូវការ


In [ ]:
# Install required packages
!pip install openai foundry-local-sdk

## នាំចូលបណ្ណាល័យ និង ការកំណត់


In [ ]:
import os
import sys
from openai import OpenAI
import time
from typing import Tuple

try:
    from foundry_local import FoundryLocalManager
    FOUNDRY_SDK_AVAILABLE = True
    print("✅ Foundry Local SDK is available")
except ImportError:
    FOUNDRY_SDK_AVAILABLE = False
    print("⚠️ Foundry Local SDK not available, manual configuration will be used")

## ជម្រើសការកំណត់

ជ្រើសរើសរវាង Azure OpenAI (ពពក) ឬ Foundry Local (នៅលើឧបករណ៍) ដោយកំណត់អថេរបរិយាកាសដែលសមរម្យ។


### ជម្រើស 1: ការកំណត់ Azure OpenAI

ដកសញ្ញា comment ហើយកំណត់ព័ត៌មានសម្គាល់ Azure OpenAI របស់អ្នក:


In [ ]:
# Azure OpenAI Configuration
# Uncomment and set your actual values

# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource.openai.azure.com"
# os.environ["AZURE_OPENAI_API_KEY"] = "your-api-key-here"
# os.environ["AZURE_OPENAI_API_VERSION"] = "2024-08-01-preview"
# os.environ["MODEL"] = "your-deployment-name"  # e.g., "gpt-4"

print("Azure OpenAI configuration ready (if credentials are set)")

### ជម្រើសទី 2: ការកំណត់ក្នុង​តំបន់​របស់ Foundry

ការកំណត់លំនាំដើមសម្រាប់ការសន្មត់ក្នុង​តំបន់:


In [ ]:
# Foundry Local Configuration (default)
FOUNDRY_MODEL = "phi-4-mini"  # Change to your preferred model
FOUNDRY_BASE_URL = "http://localhost:51211"
FOUNDRY_API_KEY = ""  # Usually empty for local

print(f"Foundry Local configuration ready with model: {FOUNDRY_MODEL}")

## មុខងារបង្កើត client

មុខងារទាំងនេះបង្កើត client OpenAI ដែលសមរម្យដោយផ្អែកលើការកំណត់របស់អ្នក:


In [ ]:
def create_azure_client() -> Tuple[OpenAI, str]:
    """Create Azure OpenAI client."""
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    azure_api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    
    if not azure_endpoint or not azure_api_key:
        raise ValueError("Azure OpenAI endpoint and API key are required")
    
    model = os.environ.get("MODEL", "your-deployment-name")
    client = OpenAI(
        base_url=f"{azure_endpoint}/openai",
        api_key=azure_api_key,
        default_query={"api-version": azure_api_version},
    )
    
    print(f"🌐 Azure OpenAI client created with model: {model}")
    return client, model


def create_foundry_client() -> Tuple[OpenAI, str]:
    """Create Foundry Local client with SDK management."""
    alias = FOUNDRY_MODEL
    
    if FOUNDRY_SDK_AVAILABLE:
        try:
            # Use FoundryLocalManager for proper service management
            print(f"🔄 Initializing Foundry Local with model: {alias}...")
            manager = FoundryLocalManager(alias)
            model_info = manager.get_model_info(alias)
            
            # Configure OpenAI client to use local Foundry service
            client = OpenAI(
                base_url=manager.endpoint,
                api_key=manager.api_key  # API key is not required for local usage
            )
            
            print(f"✅ Foundry Local SDK initialized")
            print(f"   Endpoint: {manager.endpoint}")
            print(f"   Model: {model_info.id}")
            return client, model_info.id
        except Exception as e:
            print(f"⚠️ Could not use Foundry SDK ({e}), falling back to manual configuration")
    
    # Fallback to manual configuration
    client = OpenAI(
        base_url=f"{FOUNDRY_BASE_URL}/v1",
        api_key=FOUNDRY_API_KEY
    )
    
    print(f"🔧 Manual Foundry Local configuration")
    print(f"   Endpoint: {FOUNDRY_BASE_URL}/v1")
    print(f"   Model: {alias}")
    return client, alias

## ចាប់ផ្ដើម Client

វាស្វ័យប្រវត្តិនឹងស្គាល់ថាតើត្រូវប្រើ Azure OpenAI ឬ Foundry Local:


In [ ]:
def initialize_client() -> Tuple[OpenAI, str, str]:
    """Initialize the appropriate OpenAI client."""
    
    # Check for Azure OpenAI configuration
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    
    if azure_endpoint and azure_api_key:
        print("🌐 Azure OpenAI configuration detected")
        try:
            client, model = create_azure_client()
            return client, model, "azure"
        except Exception as e:
            print(f"❌ Azure OpenAI initialization failed: {e}")
            print("🔄 Falling back to Foundry Local...")
    
    # Use Foundry Local
    print("🏠 Using Foundry Local configuration")
    try:
        client, model = create_foundry_client()
        return client, model, "foundry"
    except Exception as e:
        print(f"❌ Foundry Local initialization failed: {e}")
        raise

# Initialize the client
print("Initializing OpenAI client...")
print("=" * 50)
client, model, provider = initialize_client()
print("=" * 50)
print(f"✅ Initialization complete! Using {provider} with model: {model}")

## ការបញ្ចប់ការជជែកមូលដ្ឋាន

សាកល្បងការបញ្ចប់ការជជែកសាមញ្ញ:


In [ ]:
def simple_chat(prompt: str, max_tokens: int = 150) -> str:
    """Send a simple chat message and get response."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# Test basic chat
test_prompt = "Say hello from the SDK quickstart and explain what you are in one sentence."

print(f"👤 User: {test_prompt}")
print("\n🤖 Assistant:")
response = simple_chat(test_prompt)
print(response)

## ការបញ្ចប់សន្ទនាដោយស្ទ្រីម

បង្ហាញការឆ្លើយតបជាស្ទ្រីម ដើម្បីផ្តល់បទពិសោធន៍ប្រើប្រាស់ល្អប្រសើរ:


In [ ]:
def streaming_chat(prompt: str, max_tokens: int = 300) -> str:
    """Send a chat message with streaming response."""
    try:
        print("🤖 Assistant (streaming):")
        
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            stream=True
        )
        
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_response += content
        
        print("\n")  # New line after streaming
        return full_response
    except Exception as e:
        error_msg = f"Error: {e}"
        print(error_msg)
        return error_msg

# Test streaming chat
streaming_prompt = "Explain the key benefits of using Microsoft Foundry Local for AI development. Include aspects like privacy, performance, and cost."

print(f"👤 User: {streaming_prompt}\n")
streaming_response = streaming_chat(streaming_prompt)

## សន្ទនាច្រើនជំហាន

បង្ហាញពីការរក្សាបរិបទនៃសន្ទនា:


In [ ]:
class ConversationManager:
    """Manages multi-turn conversations with context."""
    
    def __init__(self, system_prompt: str = None):
        self.messages = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})
    
    def send_message(self, user_message: str, max_tokens: int = 200) -> str:
        """Send a message and get response while maintaining context."""
        # Add user message to conversation
        self.messages.append({"role": "user", "content": user_message})
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=self.messages,
                max_tokens=max_tokens
            )
            
            assistant_message = response.choices[0].message.content
            
            # Add assistant response to conversation
            self.messages.append({"role": "assistant", "content": assistant_message})
            
            return assistant_message
        except Exception as e:
            return f"Error: {e}"
    
    def get_conversation_length(self) -> int:
        """Get the number of messages in the conversation."""
        return len(self.messages)

# Create conversation manager with system prompt
system_prompt = "You are a helpful AI assistant specialized in explaining AI and machine learning concepts. Be concise but informative."
conversation = ConversationManager(system_prompt)

# Multi-turn conversation example
conversation_turns = [
    "What is the difference between AI inference on-device vs in the cloud?",
    "Which approach is better for privacy?",
    "What about performance and latency considerations?"
]

for i, turn in enumerate(conversation_turns, 1):
    print(f"\n{'='*60}")
    print(f"Turn {i}")
    print(f"{'='*60}")
    print(f"👤 User: {turn}")
    
    response = conversation.send_message(turn)
    print(f"\n🤖 Assistant: {response}")

print(f"\n📊 Conversation summary: {conversation.get_conversation_length()} messages total")

## ការប្រៀបធៀបសមត្ថភាព

ប្រៀបធៀបពេលវេលាចម្លើយសម្រាប់សេណារីយ៉ូផ្សេងៗ៖


In [ ]:
def benchmark_response_time(prompt: str, iterations: int = 3) -> dict:
    """Benchmark response time for a given prompt."""
    times = []
    responses = []
    
    for i in range(iterations):
        start_time = time.time()
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=50  # Keep responses short for timing
            )
            
            end_time = time.time()
            response_time = end_time - start_time
            
            times.append(response_time)
            responses.append(response.choices[0].message.content)
            
        except Exception as e:
            print(f"Error in iteration {i+1}: {e}")
    
    if times:
        avg_time = sum(times) / len(times)
        min_time = min(times)
        max_time = max(times)
        
        return {
            "average_time": avg_time,
            "min_time": min_time,
            "max_time": max_time,
            "all_times": times,
            "sample_response": responses[0] if responses else None
        }
    
    return {"error": "No successful responses"}

# Benchmark different types of prompts
benchmark_prompts = [
    "What is AI?",
    "Explain machine learning in simple terms.",
    "List 3 benefits of edge computing."
]

print(f"⏱️  Performance Benchmark ({provider} - {model})")
print("=" * 60)

for prompt in benchmark_prompts:
    print(f"\n📝 Prompt: '{prompt}'")
    results = benchmark_response_time(prompt)
    
    if "error" not in results:
        print(f"   ⏰ Average time: {results['average_time']:.2f}s")
        print(f"   ⚡ Fastest: {results['min_time']:.2f}s")
        print(f"   🐌 Slowest: {results['max_time']:.2f}s")
        print(f"   📄 Sample response: {results['sample_response'][:100]}...")
    else:
        print(f"   ❌ {results['error']}")

## ការកំណត់កម្រិតខ្ពស់ និងការដោះស្រាយកំហុស

សាកល្បងប៉ារ៉ាម៉ែត្រ និងស្ថានភាពកំហុសផ្សេងៗ៖


In [ ]:
def test_different_parameters():
    """Test chat completions with different parameters."""
    prompt = "Write a creative short story about AI."
    
    # Test different temperature values
    temperatures = [0.1, 0.5, 0.9]
    
    for temp in temperatures:
        print(f"\n🌡️ Temperature: {temp}")
        print("-" * 30)
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                temperature=temp
            )
            
            print(f"Response: {response.choices[0].message.content[:150]}...")
            
        except Exception as e:
            print(f"Error with temperature {temp}: {e}")

test_different_parameters()

## ការត្រួតពិនិត្យសុខភាពសេវា

ការត្រួតពិនិត្យសុខភាព និងសមត្ថភាពសេវាដោយទូលំទូលាយ:


In [ ]:
def comprehensive_health_check():
    """Perform comprehensive health check of the service."""
    print("🏥 Comprehensive Health Check")
    print("=" * 50)
    
    # 1. Check model listing
    try:
        models_response = client.models.list()
        available_models = [m.id for m in models_response.data]
        print(f"✅ Model listing: SUCCESS")
        print(f"   📋 Available models: {available_models}")
        
        if model in available_models:
            print(f"   ✅ Current model '{model}' is available")
        else:
            print(f"   ⚠️ Current model '{model}' not found in available models")
    except Exception as e:
        print(f"❌ Model listing: FAILED - {e}")
    
    # 2. Test basic completion
    try:
        test_response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Say 'Health check successful'"}],
            max_tokens=10
        )
        print(f"✅ Basic completion: SUCCESS")
        print(f"   💬 Response: {test_response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Basic completion: FAILED - {e}")
    
    # 3. Test streaming
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Count to 3"}],
            max_tokens=20,
            stream=True
        )
        
        stream_content = ""
        chunk_count = 0
        for chunk in stream:
            if chunk.choices[0].delta.content:
                stream_content += chunk.choices[0].delta.content
                chunk_count += 1
        
        print(f"✅ Streaming: SUCCESS")
        print(f"   📦 Chunks received: {chunk_count}")
        print(f"   💬 Streamed content: {stream_content.strip()}")
    except Exception as e:
        print(f"❌ Streaming: FAILED - {e}")
    
    # 4. Provider-specific information
    print(f"\n📊 Configuration Summary:")
    print(f"   🏢 Provider: {provider}")
    print(f"   🤖 Model: {model}")
    if provider == "foundry":
        print(f"   🏠 Foundry SDK Available: {FOUNDRY_SDK_AVAILABLE}")
        print(f"   🔗 Base URL: {FOUNDRY_BASE_URL}")
    elif provider == "azure":
        print(f"   🌐 Azure Endpoint: {os.environ.get('AZURE_OPENAI_ENDPOINT', 'Not set')}")
        print(f"   🔑 API Version: {os.environ.get('AZURE_OPENAI_API_VERSION', 'Not set')}")

comprehensive_health_check()

## ការធ្វើតេស្តអន្តរកម្ម

ប្រើកោសិកានេះដើម្បីសាកល្បងសំណើររបស់អ្នកដោយអន្តរកម្ម:


In [ ]:
# Interactive testing - modify the prompt below
custom_prompt = "Explain the concept of 'edge AI' and why it's becoming important."
use_streaming = True  # Set to False for regular completion

print(f"👤 Custom Prompt: {custom_prompt}\n")

if use_streaming:
    custom_response = streaming_chat(custom_prompt, max_tokens=250)
else:
    custom_response = simple_chat(custom_prompt, max_tokens=250)
    print(f"🤖 Assistant: {custom_response}")

## សេចក្ដីសង្ខេប និងជំហានបន្ទាប់

កំណត់ត្រានេះបានបង្ហាញអំពីការរួមបញ្ចូល OpenAI SDK ជាស្ថានភាពកម្រិតខ្ពស់ជាមួយ៖

### ✅ លក្ខណៈពិសេសសំខាន់ដែលបានគ្របដណ្តប់

1. **ការគាំទ្រក្រុមអ្នកផ្គត់ផ្គង់ច្រើន**: ការប្តូរយ៉ាងរលូនរវាង Azure OpenAI និង Foundry Local
2. **ការឆ្លើយតបដោយបញ្ចូលចេញ (Streaming Responses)**: ការបង្កើត token ក្នុងពេលពិត សម្រាប់ UX ដែលប្រសើរជាង
3. **ការគ្រប់គ្រងសន្ទនា**: សន្ទនាច្រើនជុំជាមួយបរិបទ
4. **ការវាស់ស្ទង់កម្រិតសមត្ថភាព**: ការវាស់ពេលឆ្លើយតប និងវិភាគ
5. **ការត្រួតពិនិត្យសុខភាពទាំងមូល**: ការផ្ទៀងផ្ទាត់សេវាកម្ម និងវិភាគបញ្ហា
6. **ការដោះស្រាយកំហុស**: ការដោះស្រាយកំហុសដែលរឹងមាំ និងយន្តការជំនួស

### 🏆 Foundry Local ប្រៀបធៀបនឹង Azure OpenAI

| ផ្នែក | Foundry Local | Azure OpenAI |
|--------|---------------|---------------|
| **ភាពឯកជន** | ✅ ទិន្នន័យនៅក្នុងស្រុក | ⚠️ ទិន្នន័យត្រូវបានផ្ញើទៅពពក |
| **ភាពយឺតនៃការឆ្លើយតប** | ✅ ទាប (inference ក្នុងស្រុក) | ⚠️ ខ្ពស់ (ពឹងផ្អែកលើបណ្ដាញ) |
| **ថ្លៃប្រាក់** | ✅ មិនគិតថ្លៃ (បន្ទាប់ពីមានឧបករណ៍) | 💰 បង់តាម token |
| **អត់អ៊ីនធឺណិត (Offline)** | ✅ ដំណើរការបានដោយគ្មានអ៊ីនធឺណិត | ❌ តម្រូវអ៊ីនធឺណិត |
| **ភាពចម្រុះនៃម៉ូឌែល** | ⚠️ ជម្រើសមានកំណត់ | ✅ ចូលដំណើរការម៉ូឌែលទាំងមូល |
| **ការពង្រីក** | ⚠️ ពឹងផ្អែកលើឧបករណ៍ | ✅ អាចពង្រីកបានដោយគ្មានដែនកំណត់ |

### 🚀 ជំហានបន្ទាប់

- **Sample 04**: ការសង់កម្មវិធីសន្ទនា Chainlit
- **Sample 05**: ប្រព័ន្ធរៀបចំកម្មវិធីភ្នាក់ងារច្រើន
- **Sample 06**: ការបញ្ជូនម៉ូឌែលដោយឆ្លាតវៃ
- **Production Deployment**: ការពិចារណាអំពីការពង្រីក និងការត្រួតពិនិត្យ

### 💡 ការអនុវត្តល្អបំផុត

1. **តែងតែអនុវត្តយន្តការជំនួស** រវាងអ្នកផ្គត់ផ្គង់
2. **ប្រើស្ទ្រីមមิ่งសម្រាប់ចម្លើយវែងៗ** ដើម្បីធ្វើឲ្យបទពិសោធន៍ក្នុងការប្រើប្រាស់ប្រសើរឡើង
3. **អនុវត្តការដោះស្រាយកំហុសយ៉ាងត្រឹមត្រូវ** សម្រាប់កម្មវិធីក្នុងផលិតកម្ម
4. **តាមដានពេលឆ្លើយតប និងចំណាយ** សម្រាប់អ្នកផ្គត់ផ្គង់ដែលខុសគ្នា
5. **ជ្រើសរើសអ្នកផ្គត់ផ្គង់ដែលត្រឹមត្រូវ** ដោយផ្អែកលើតម្រូវការពិសេសរបស់អ្នក


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែបញ្ញាសិប្បនិម្មិត [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលដែលយើងខំប្រឹងសម្រេចភាពត្រឹមត្រូវ សូមយល់ឲ្យបានថាការបកប្រែដោយស្វ័យប្រវត្តិនេះអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវបានបញ្ចូល។ ឯកសារដើមនៅក្នុងភាសាដើមគួរត្រូវបានចាត់ទុកជាប្រភពផ្លូវការដែលមានអាទិភាព។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងផ្តល់អនុសាសន៍ឱ្យពិចារណាការបកប្រែដោយអ្នកបកប្រែវិជ្ជាជីវៈមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកប្រែឆ្លុះបញ្ចាំងខុស ដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
